# Detected Intensity Above A Modeled PFB Bandpass

This notebook targets the central science question: if the
original voltage-domain signal has fixed amplitude, how much
detected spectrogram power rises above the local background at
different positions in a coarse channel?

We inject a zero-drift tone centered on exact fine-channel bins
across one coarse channel. Every tone has the same original
voltage amplitude. For each noisy realization, we fit a scaled
ideal PFB bandpass to the final flattened spectrum while
masking the injected channel neighborhood. The signal
measurement is the residual above that modeled bandpass.

This keeps the measurement on the actual final-frequency axis:
there is no same-fine-bin cross-coarse summing. The optional
aperture measurement is only a local final-frequency window
around the injected channel.


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display

from pfb_response_tools import (
    PFBExperimentConfig,
    detected_intensity_sweep,
    ideal_response,
    local_noise_stats,
    modeled_bandpass_excess_sweep,
    modeled_bandpass_summary,
    noise_overlay_summary,
    normalize_column,
    run_spectrogram,
    tone_response_sweep,
)

plt.rcParams.update({
    "figure.figsize": (9, 4),
    "axes.grid": True,
    "grid.alpha": 0.25,
})


In [ ]:
config = PFBExperimentConfig(spectra_factor=32)
target_coarse_offset = 1
num_chans = 3

edge_bins = [
    0, 1, 2, 4, 8, 16, 32, 64,
    96, 128, 160, 192, 224, 240, 248, 252, 254, 255,
]

bandpass_check = modeled_bandpass_summary(
    config,
    seed=20260502,
    num_chans=num_chans,
)

rows = modeled_bandpass_excess_sweep(
    config,
    edge_bins,
    tone_level=0.02,
    target_coarse_offset=target_coarse_offset,
    num_chans=num_chans,
    aperture_half_width=3,
    fit_guard_bins=8,
    num_trials=4,
    reference_bin=config.fftlength // 2,
)

bandpass_check, rows[:3], rows[-3:]


In [ ]:
noise_data = run_spectrogram(
    config,
    seed=20260502,
    noise=True,
    num_chans=num_chans,
)
noise_spectrum = noise_data.mean(axis=0)
response = ideal_response(config, num_chans=num_chans)
response_model = noise_spectrum.mean() * response
response_model *= noise_spectrum.mean() / response_model.mean()

final_frequency_index = np.arange(len(noise_spectrum))
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(final_frequency_index, noise_spectrum / noise_spectrum.mean(), lw=1.2, label="Noise mean")
ax.plot(final_frequency_index, response_model / response_model.mean(), lw=2, alpha=0.8, label="Scaled ideal PFB model")
for boundary in range(config.fftlength, len(noise_spectrum), config.fftlength):
    ax.axvline(boundary, color="0.25", lw=0.8, alpha=0.4)
ax.set_xlabel("Final flattened frequency-channel index")
ax.set_ylabel("Relative power")
ax.set_title("Noise baseline follows the repeated PFB bandpass")
ax.legend()
display(fig)
plt.close(fig)


In [ ]:
fine_offset_khz = np.asarray([row["fine_offset_hz"] for row in rows]) / 1e3
center_row = rows[edge_bins.index(config.fftlength // 2)]
center_peak = center_row["peak_excess_power"]
center_aperture = center_row["aperture_excess_power"]

fig, ax = plt.subplots()
ax.plot(
    fine_offset_khz,
    [row["ideal_response_relative_to_center"] for row in rows],
    "o-",
    label="Ideal PFB response at target bin",
)
ax.errorbar(
    fine_offset_khz,
    [row["peak_excess_relative_to_center"] for row in rows],
    yerr=[row["peak_excess_sem"] / center_peak for row in rows],
    fmt="o-",
    capsize=3,
    label="Peak-channel excess above modeled bandpass",
)
ax.errorbar(
    fine_offset_khz,
    [row["aperture_excess_relative_to_center"] for row in rows],
    yerr=[row["aperture_excess_sem"] / center_aperture for row in rows],
    fmt="o-",
    capsize=3,
    label="+/-3 final-channel aperture excess above model",
)
ax.axhline(1, color="0.25", lw=1, alpha=0.5)
ax.set_xlabel("Fine-channel offset from coarse-channel center (kHz)")
ax.set_ylabel("Relative to center-bin detected excess")
ax.set_title("Same original tone amplitude: excess above modeled bandpass")
ax.legend()
display(fig)
plt.close(fig)


In [ ]:
fig, ax = plt.subplots()
ax.errorbar(
    fine_offset_khz,
    [row["path_snr_relative_to_center"] for row in rows],
    yerr=[row["peak_excess_sem"] / center_peak for row in rows],
    fmt="o-",
    capsize=3,
    label="Peak-bin path SNR",
)
ax.plot(
    fine_offset_khz,
    [row["local_noise_std"] / center_row["local_noise_std"] for row in rows],
    "o-",
    label="Local noise std",
)
ax.plot(
    fine_offset_khz,
    [row["relative_residual_rms"] / center_row["relative_residual_rms"] for row in rows],
    "o-",
    label="Bandpass residual RMS",
)
ax.axhline(1, color="0.25", lw=1, alpha=0.5)
ax.set_xlabel("Fine-channel offset from coarse-channel center (kHz)")
ax.set_ylabel("Relative to center")
ax.set_title("Modeled-bandpass excess, local noise, and fit quality")
ax.legend()
display(fig)
plt.close(fig)


In [ ]:
compact = [
    {
        "fine_bin": row["fine_bin"],
        "offset_kHz": row["fine_offset_hz"] / 1e3,
        "ideal_rel": row["ideal_response_relative_to_center"],
        "peak_excess_rel": row["peak_excess_relative_to_center"],
        "peak_excess_sem_rel": row["peak_excess_sem"] / center_peak,
        "aperture_excess_rel": row["aperture_excess_relative_to_center"],
        "path_snr_rel": row["path_snr_relative_to_center"],
    }
    for row in rows
]
compact


This is the distinction we need for inference. The fixed
original tone does not retain the same detected intensity across
the coarse channel. The measurement here is exactly the final
spectrum quantity we care about: data minus a modeled PFB
bandpass baseline. In the middle of the coarse channel, the
excess approximately follows the ideal PFB response. Near the
edge, the peak-channel excess can fall below that smooth model,
and the local final-frequency aperture tells us whether the
detector recovers power spread into adjacent final channels.

For real observations this means a narrowband candidate near a
coarse-channel edge cannot be interpreted with the same
intrinsic-intensity calibration as one near the coarse-channel
center. The first useful correction is likely a PFB-position
transfer curve for synthetic voltage tones, with separate
treatment for peak-bin detection and integrated-window
detection.
